In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import os
import shutil
import zipfile

INPUT_ROOT = "/kaggle/input"
WORKING_DIR = "/kaggle/working"

bench_source = None
zip_source = None

for root, directories, files in os.walk(INPUT_ROOT):
    for filename in files:
        full_path = os.path.join(root, filename)

        if filename == "bench.py":
            bench_source = full_path

        if filename == "material (6).zip":
            zip_source = full_path

assert bench_source is not None, "bench.py was not found"
assert zip_source is not None, "material (6).zip was not found"

shutil.copy(
    bench_source,
    os.path.join(WORKING_DIR, "bench.py")
)

with zipfile.ZipFile(zip_source, "r") as archive:
    archive.extractall(WORKING_DIR)

required_files = [
    "bench.py",
    "prompts.txt",
    "capacity-note.md",
    "verify_cell.py"
]

for filename in required_files:
    path = os.path.join(WORKING_DIR, filename)
    print(filename, "FOUND" if os.path.exists(path) else "MISSING")

print("LAB FILES READY")

In [9]:
import os

for root, directories, files in os.walk("/kaggle/input"):
    for filename in files:
        print(os.path.join(root, filename))

/kaggle/input/datasets/wasanali/w3d5-main/bench.py
/kaggle/input/datasets/wasanali/w3d5-main/material (6)/capacity-note.md
/kaggle/input/datasets/wasanali/w3d5-main/material (6)/verify_cell.py
/kaggle/input/datasets/wasanali/w3d5-main/material (6)/prompts.txt


In [8]:
import os
import shutil

SOURCE_DIR = "/kaggle/input/datasets/wasanali/w3d5-main"
MATERIAL_DIR = os.path.join(
    SOURCE_DIR,
    "material (6)"
)
WORKING_DIR = "/kaggle/working"

source_files = {
    "bench.py": os.path.join(
        SOURCE_DIR,
        "bench.py"
    ),
    "prompts.txt": os.path.join(
        MATERIAL_DIR,
        "prompts.txt"
    ),
    "capacity-note.md": os.path.join(
        MATERIAL_DIR,
        "capacity-note.md"
    ),
    "verify_cell.py": os.path.join(
        MATERIAL_DIR,
        "verify_cell.py"
    )
}

for filename, source_path in source_files.items():
    destination = os.path.join(
        WORKING_DIR,
        filename
    )

    shutil.copy(source_path, destination)
    print(filename, "FOUND")

os.chdir(WORKING_DIR)

print("LAB FILES READY")

bench.py FOUND
prompts.txt FOUND
capacity-note.md FOUND
verify_cell.py FOUND
LAB FILES READY


In [7]:
import subprocess
import sys

packages = [
    "vllm==0.6.*",
    "transformers==4.46.*",
    "accelerate==1.1.*",
    "autoawq==0.2.*",
    "httpx==0.27.*",
    "openai==1.54.*"
]

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        *packages
    ],
    check=True
)

print("INSTALLATION COMPLETE")
print("RESTART SESSION NOW")

INSTALLATION COMPLETE
RESTART SESSION NOW


In [10]:
import os
import subprocess
import sys

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
SERVER_LOG = "/kaggle/working/w3d5_server.log"

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open(SERVER_LOG, "w")

server = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL_ID,
        "--dtype",
        "half",
        "--max-model-len",
        "4096",
        "--gpu-memory-utilization",
        "0.85",
        "--quantization",
        "awq",
        "--enable-auto-tool-choice",
        "--tool-call-parser",
        "hermes",
        "--disable-frontend-multiprocessing",
        "--port",
        "8000"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
    env=environment
)

print("LOCKED SERVER LAUNCHED")
print("MODEL:", MODEL_ID)
print("PID:", server.pid)
print("LOG:", SERVER_LOG)

LOCKED SERVER LAUNCHED
MODEL: Qwen/Qwen2.5-1.5B-Instruct-AWQ
PID: 197
LOG: /kaggle/working/w3d5_server.log


In [11]:
import time
import httpx

MODELS_URL = "http://localhost:8000/v1/models"

print("Waiting for the locked model...")

for attempt in range(100):
    try:
        response = httpx.get(
            MODELS_URL,
            timeout=10.0
        )

        if response.status_code == 200:
            print("SERVER HEALTHY:", response.status_code)
            print("MODEL:", response.json()["data"][0]["id"])
            break

    except Exception:
        pass

    if server.poll() is not None:
        print("SERVER FAILED")

        with open(
            "/kaggle/working/w3d5_server.log",
            "r",
            errors="replace"
        ) as file:
            print(file.read()[-6000:])

        break

    time.sleep(3)

else:
    print("SERVER TIMEOUT")

    with open(
        "/kaggle/working/w3d5_server.log",
        "r",
        errors="replace"
    ) as file:
        print(file.read()[-6000:])

Waiting for the locked model...
SERVER HEALTHY: 200
MODEL: Qwen/Qwen2.5-1.5B-Instruct-AWQ


In [12]:
PREDICTED_KNEE = 8
TARGET_P95_S = 8.0

prediction_card = f"""# W3D5 Prediction Card

- Locked model: {MODEL_ID}
- Predicted knee concurrency: {PREDICTED_KNEE}
- Target p95 SLO: {TARGET_P95_S} seconds
"""

with open(
    "/kaggle/working/prediction-card.md",
    "w",
    encoding="utf-8"
) as file:
    file.write(prediction_card)

print(prediction_card)
print("PREDICTION CARD CREATED")

# W3D5 Prediction Card

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Predicted knee concurrency: 8
- Target p95 SLO: 8.0 seconds

PREDICTION CARD CREATED


In [13]:
import subprocess
import sys

command = [
    sys.executable,
    "/kaggle/working/bench.py",
    "--base-url",
    "http://localhost:8000",
    "--model",
    MODEL_ID,
    "--concurrency",
    "1,2,4,8,16",
    "--requests-per-level",
    "20",
    "--prompt-file",
    "/kaggle/working/prompts.txt",
    "--out",
    "/kaggle/working/bench_report.json"
]

result = subprocess.run(
    command,
    text=True
)

print("BENCHMARK EXIT CODE:", result.returncode)

if result.returncode == 0:
    print("BENCHMARK SWEEP COMPLETE")
else:
    print("BENCHMARK FAILED")

[level 1] tok/s=86.81 ttft_p95=0.0577 errors=0
[level 2] tok/s=154.54 ttft_p95=0.0865 errors=0
[level 4] tok/s=268.52 ttft_p95=0.1317 errors=0
[level 8] tok/s=435.22 ttft_p95=0.183 errors=0
[level 16] tok/s=635.96 ttft_p95=0.2794 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     86.81      0.055      0.058     1.488    20     0
   2    154.54      0.064      0.086     1.587    20     0
   4    268.52      0.065      0.132     1.772    20     0
   8    435.22      0.153      0.183     2.095    20     0
  16    635.96      0.277      0.279     2.606    20     0

wrote /kaggle/working/bench_report.json (run appended)
BENCHMARK EXIT CODE: 0
BENCHMARK SWEEP COMPLETE


In [14]:
import subprocess
import sys

command = [
    sys.executable,
    "/kaggle/working/bench.py",
    "--base-url",
    "http://localhost:8000",
    "--model",
    MODEL_ID,
    "--concurrency",
    "1,2,4,8,16,32",
    "--requests-per-level",
    "20",
    "--prompt-file",
    "/kaggle/working/prompts.txt",
    "--out",
    "/kaggle/working/bench_report.json"
]

result = subprocess.run(
    command,
    text=True
)

print("BENCHMARK EXIT CODE:", result.returncode)

if result.returncode == 0:
    print("EXTENDED BENCHMARK COMPLETE")
else:
    print("EXTENDED BENCHMARK FAILED")

[level 1] tok/s=79.67 ttft_p95=0.0625 errors=0
[level 2] tok/s=155.68 ttft_p95=0.0784 errors=0
[level 4] tok/s=268.09 ttft_p95=0.1174 errors=0
[level 8] tok/s=440.89 ttft_p95=0.1777 errors=0
[level 16] tok/s=683.7 ttft_p95=0.2696 errors=0
[level 32] tok/s=729.52 ttft_p95=0.3131 errors=0

conc     tok/s   ttft_p50   ttft_p95   lat_p95    ok   err
----------------------------------------------------------
   1     79.67      0.058      0.062     1.625    20     0
   2    155.68      0.066      0.078     1.619    20     0
   4    268.09      0.066      0.117     1.784    20     0
   8    440.89      0.157      0.178     2.093    20     0
  16    683.70      0.268      0.270     2.532    20     0
  32    729.52      0.310      0.313     2.844    20     0

wrote /kaggle/working/bench_report.json (run appended)
BENCHMARK EXIT CODE: 0
EXTENDED BENCHMARK COMPLETE


In [15]:
import json

BENCH_PATH = "/kaggle/working/bench_report.json"
KNEE_PATH = "/kaggle/working/knee.json"
CAPACITY_PATH = "/kaggle/working/capacity-note.md"

TARGET_P95_S = 8.0

with open(BENCH_PATH, "r", encoding="utf-8") as file:
    benchmark = json.load(file)

levels = benchmark["runs"][-1]["levels"]

under_target = [
    level
    for level in levels
    if level["latency_p95_s"] <= TARGET_P95_S
]

knee = max(
    under_target,
    key=lambda level: level["concurrency"]
)

highest_tested = max(
    level["concurrency"]
    for level in levels
)

sweep_bounded = (
    knee["concurrency"] == highest_tested
)

request_rate = round(
    knee["ok"] / knee["wall_s"],
    2
)

knee_report = {
    "target_p95_s": TARGET_P95_S,
    "knee_concurrency": knee["concurrency"],
    "sweep_bounded": sweep_bounded,
    "tokens_per_s_at_knee": knee["tokens_per_s"],
    "latency_p95_s_at_knee": knee["latency_p95_s"],
    "max_sustainable_request_rate": request_rate
}

with open(KNEE_PATH, "w", encoding="utf-8") as file:
    json.dump(knee_report, file, indent=2)

capacity_note = f"""# Capacity Note

## The numbers

- Locked model: {MODEL_ID}
- Target p95 end-to-end latency: {TARGET_P95_S} seconds
- Knee concurrency: {knee["concurrency"]} (sweep-bounded)
- Tokens per second at the knee: {knee["tokens_per_s"]}
- p95 latency at the knee: {knee["latency_p95_s"]} seconds
- Max sustainable request rate: {request_rate} requests per second
- Request errors: {knee["errors"]}

## The limiting family

- Memory-bound tendency: throughput increased only slightly from concurrency 16 to 32 while p95 latency continued to rise, which is consistent with a decode memory-bandwidth ceiling.

## Why the knee, not the peak

- The knee is the highest tested concurrency that still satisfies the latency SLO, so it represents capacity the service can honestly promise rather than throughput alone.
"""

with open(
    CAPACITY_PATH,
    "w",
    encoding="utf-8"
) as file:
    file.write(capacity_note)

print(json.dumps(knee_report, indent=2))
print()
print(capacity_note)
print("KNEE AND CAPACITY NOTE CREATED")

{
  "target_p95_s": 8.0,
  "knee_concurrency": 32,
  "sweep_bounded": true,
  "tokens_per_s_at_knee": 729.52,
  "latency_p95_s_at_knee": 2.8438,
  "max_sustainable_request_rate": 7.03
}

# Capacity Note

## The numbers

- Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
- Target p95 end-to-end latency: 8.0 seconds
- Knee concurrency: 32 (sweep-bounded)
- Tokens per second at the knee: 729.52
- p95 latency at the knee: 2.8438 seconds
- Max sustainable request rate: 7.03 requests per second
- Request errors: 0

## The limiting family

- Memory-bound tendency: throughput increased only slightly from concurrency 16 to 32 while p95 latency continued to rise, which is consistent with a decode memory-bandwidth ceiling.

## Why the knee, not the peak

- The knee is the highest tested concurrency that still satisfies the latency SLO, so it represents capacity the service can honestly promise rather than throughput alone.

KNEE AND CAPACITY NOTE CREATED


In [16]:
import os

os.chdir("/kaggle/working")

with open(
    "/kaggle/working/verify_cell.py",
    "r",
    encoding="utf-8"
) as file:
    verifier_code = file.read()

exec(
    compile(
        verifier_code,
        "verify_cell.py",
        "exec"
    )
)

levels: 6, concurrencies: [1, 2, 4, 8, 16, 32], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [17]:
import base64
import os
import zipfile
from IPython.display import HTML, display

files_to_download = [
    "/kaggle/working/bench_report.json",
    "/kaggle/working/knee.json",
    "/kaggle/working/capacity-note.md",
    "/kaggle/working/prediction-card.md"
]

zip_path = "/kaggle/working/w3d5-main-artifacts.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as archive:
    for file_path in files_to_download:
        archive.write(
            file_path,
            arcname=os.path.basename(file_path)
        )

with open(zip_path, "rb") as file:
    encoded = base64.b64encode(file.read()).decode()

download_link = f"""
<a download="w3d5-main-artifacts.zip"
   href="data:application/zip;base64,{encoded}">
   Download W3D5 Main Artifacts
</a>
"""

display(HTML(download_link))